In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
TOPIC_DIR = PROJECT_ROOT / '03_Pathomics/00_Preprocessing'
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))


## Key2

对于病理的任务，将病理的所有tiles转化成histogram或者tfidf的patient特征。

![](http://www.medai.icu/storage/attachments/2022/06/26/n41q4HeDjvIOZnyfoKH28c5YNioGdB7OZwO35XOf.png)

参考论文: [Development and interpretation of a pathomics-based model for the prediction of microsatellite instability in Colorectal Cancer](http://www.medai.icu/download?url=http://www.medai.icu/apiv3/attachment.download?sign=1667478d908313ae1e01543e229d02de&attachmentsId=1061&threadId=230)

In [ ]:
import re

def id_map(x):
    if x.startswith('TCGA') :
        return x[:12] 
    else:
        items = re.split('[ |\-|_]', x)
        return items[0]


In [ ]:
import pandas as pd
from onekey_algo.custom.utils import key2

train_log = pd.read_csv(str(EXTERNAL_INPUT_DIR / '20230206/resnet50/viz/BST_TRAIN_RESULTS.txt'), sep='\t',
                        names=['fname', 'prob', 'pred', 'gt'])

val_log = pd.read_csv(str(EXTERNAL_INPUT_DIR / '20230206/resnet50/viz/BST_VAL_RESULTS.txt'), sep='\t',
                      names=['fname', 'prob', 'pred', 'gt'])
log = pd.concat([train_log, val_log], axis=0)
log['prob'] = log['prob'].round(decimals=2)
log[['group']] = log[['fname']].applymap(id_map)
log


### 直方图

```python
def key2histogram(data: pd.DataFrame, group_column: str, histo_columns: Union[str, List[str]],
                  histo_lists: Union[list, List[list]] = None, default_value=0, norm: bool = False):
    """
    所有的数据生成直方图特征， 多个histo_columns存在是，所有的特征进行横向拼接。
    Args:
        data: 数据
        group_column: 样本分组的列明，一般为ID
        histo_columns: 用来计算直方图的列，如果为多列，则每列计算完直方图，然后特征拼接
        histo_lists: None或者与histo_columns个数相同，为自己指定特征列表
        default_value: 不存在特征时的默认值
        norm: 要不要归一化。
    Returns:

    """
```

In [ ]:
import os

os.makedirs(str(TOPIC_DIR), exist_ok=True)
results = key2.key2histogram(log, group_column='group',histo_columns='prob', norm=True)
results.to_csv(str(PROJECT_ROOT / '03_Pathomics/superwise_path_prob_histogram_reference.csv'), header=True, index=False)
display(results)

results = key2.key2histogram(log, group_column='group',histo_columns='pred', norm=True)
results.to_csv(str(PROJECT_ROOT / '03_Pathomics/superwise_path_pred_histogram_reference.csv'), header=True, index=False)
display(results)


### TF-IDF

```python
def key2tfidf(data: pd.DataFrame, group_column: str, corpus_columns: Union[str, List[str]]):
    """
    所有的数据生成直方图特征， 多个corpus_columns存在是，所有的特征进行横向拼接。
    Args:
        data: 数据
        group_column: 样本分组的列明，一般为ID
        corpus_columns: 用来计算作为语料的列明。
    Returns:

    """
```

In [ ]:
results = key2.key2tfidf(log, group_column='group',corpus_columns='prob')
results.to_csv(str(PROJECT_ROOT / '03_Pathomics/superwise_path_prob_tfidf_reference.csv'), header=True, index=False)
display(results)

results = key2.key2tfidf(log, group_column='group',corpus_columns='pred')
results.to_csv(str(PROJECT_ROOT / '03_Pathomics/superwise_path_pred_tfidf_reference.csv'), header=True, index=False)
display(results)
